In [1]:
import numpy as np

In [2]:
rng = np.random.default_rng(0)
print(rng)

Generator(PCG64)


In [4]:
N, d = 4000, 768

In [6]:
X = rng.normal(0, 0.1, (N, d))          # every room's 768 readings: mostly background noise
print(X.shape)
print(X)

(4000, 768)
[[-0.04823541 -0.00664829  0.03045495 ... -0.07146307  0.01589888
   0.17648909]
 [-0.05802542 -0.05304807 -0.00048834 ... -0.03359873  0.07738519
   0.04614904]
 [ 0.00872132 -0.05847932 -0.11716288 ...  0.09565627 -0.0165813
  -0.16695776]
 ...
 [-0.02846183 -0.13052919  0.01450271 ... -0.07231418  0.08256148
  -0.06813394]
 [-0.00591587  0.05145689 -0.01883663 ... -0.08233894 -0.02016892
  -0.04407952]
 [ 0.12411048 -0.03154803  0.03197501 ... -0.20809003  0.04899179
  -0.1057287 ]]


In [7]:
smoke = rng.integers(0, 2, N)
heat  = rng.integers(0, 2, N)
print(smoke.shape)
print(smoke)
print(heat.shape)
print(heat)

(4000,)
[1 1 0 ... 1 1 1]
(4000,)
[0 1 1 ... 0 1 0]


In [8]:
X[:, 17]  = smoke      # line 17  = smoke sensor  ("state-ish")
X[:, 402] = heat       # line 402 = heat sensor   ("bank-ish")


In [9]:
fire = smoke & heat    # what we want: 1 only if both

In [10]:
print(fire)

[0 1 0 ... 0 1 0]


In [11]:
# The Gauge: best possible 768 knobs + offset (least squares does the tuning)
Xg = np.hstack([X, np.ones((N, 1))])
print(Xg.shape)
print(Xg)

(4000, 769)
[[-4.82354095e-02 -6.64829238e-03  3.04549521e-02 ...  1.58988772e-02
   1.76489085e-01  1.00000000e+00]
 [-5.80254247e-02 -5.30480713e-02 -4.88341164e-04 ...  7.73851929e-02
   4.61490393e-02  1.00000000e+00]
 [ 8.72132157e-03 -5.84793207e-02 -1.17162879e-01 ... -1.65812972e-02
  -1.66957759e-01  1.00000000e+00]
 ...
 [-2.84618297e-02 -1.30529189e-01  1.45027127e-02 ...  8.25614806e-02
  -6.81339416e-02  1.00000000e+00]
 [-5.91586884e-03  5.14568874e-02 -1.88366331e-02 ... -2.01689165e-02
  -4.40795194e-02  1.00000000e+00]
 [ 1.24110479e-01 -3.15480273e-02  3.19750071e-02 ...  4.89917864e-02
  -1.05728702e-01  1.00000000e+00]]


In [12]:
knobs, *_ = np.linalg.lstsq(Xg, fire, rcond=None)

In [13]:
print(knobs.shape)
print(knobs)

(769,)
[ 1.27558339e-02 -1.77316175e-02 -3.05833176e-02 -1.09789263e-01
  2.16486745e-02 -1.18725016e-02 -2.81283317e-02 -5.39274239e-03
  5.68631872e-03 -3.24347899e-03 -3.54955879e-03 -2.77474610e-02
 -2.45239549e-02  2.06179029e-02 -3.48860710e-02  6.33164006e-02
 -1.85724749e-02  5.04248023e-01 -5.75512535e-02 -1.78231272e-02
 -2.38051611e-02  5.87626800e-02 -3.90349428e-02 -4.05304017e-02
 -2.29989415e-02  3.60211554e-02 -2.88661197e-02  1.96151000e-02
  2.44437187e-03  3.77477245e-02 -4.39277080e-02  2.43474318e-03
 -6.28883152e-03  2.75868934e-02  3.57435613e-02  2.32892797e-02
 -2.02463775e-02 -4.67989347e-02  2.94713292e-02 -2.91298564e-02
  2.98440732e-02  6.77313153e-03 -4.22872632e-02 -2.73070840e-02
  5.47419381e-02 -5.22038658e-02 -1.97295211e-02  1.09290285e-01
 -9.07860613e-03 -2.73993221e-02 -1.42531881e-02  5.68543390e-02
 -2.58778996e-02 -3.04061362e-02  1.07312866e-02  4.66791156e-02
  3.38186167e-02 -2.52415332e-02  3.00392689e-02  7.93008245e-02
 -1.10918192e-02  

In [ ]:
needle = Xg @ knobs # This is the prediction
print(needle.shape)
print(needle)

(4000,)
[0.22522844 0.7530552  0.27084643 ... 0.02059237 0.81869459 0.14002741]


In [17]:
for s in (0, 1):
    for h in (0, 1):
        rows = (smoke == s) & (heat == h)
        print(f"smoke={s} heat={h}  needle ≈ {needle[rows].mean():.2f}")

smoke=0 heat=0  needle ≈ -0.21
smoke=0 heat=1  needle ≈ 0.20
smoke=1 heat=0  needle ≈ 0.21
smoke=1 heat=1  needle ≈ 0.80
